# SessionRerank+ — Kaggle Notebook v18
## Full ToolBench benchmark + Gemma-2 reranker

**Requirements (one-time):**
1. Right sidebar → Settings → Accelerator → **GPU T4 x2**
2. Settings → Turn on internet
3. Add-ons → Secrets → add `HF_TOKEN` (Gemma-2 license required)
4. Run All

In [ ]:
# CELL 1: Pull latest code from GitHub
import os, subprocess, sys

REPO_URL = 'https://github.com/joyjeni/session-aware-toolbench-rerank.git'
REPO_DIR = '/kaggle/working/sr'

if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

result = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], capture_output=True, text=True)
print('Latest commit:', result.stdout.strip())

In [ ]:
# CELL 2: Install dependencies (no torchvision, adds setuptools)
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchaudio'], check=False)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'setuptools',
    'torch==2.4.1+cu121',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers>=2.7',
    'rank-bm25',
    'transformers>=4.45',
    'huggingface_hub>=0.23',
    'accelerate>=0.30',
    'bitsandbytes>=0.43',
    'scipy>=1.11',
    'scikit-learn>=1.3',
    'numpy<2.1',
], check=True)

print('[install] done')

In [ ]:
# CELL 3: Torch smoke test (separate cell to avoid stale module cache)
import importlib, sys

for k in [m for m in sys.modules if m == 'torch' or m.startswith('torch.')]:
    del sys.modules[k]
importlib.invalidate_caches()

import torch
assert torch.cuda.is_available(), 'CUDA not available — enable GPU T4 in Settings'
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))

In [ ]:
# CELL 4: Install sessionrerank package from cloned repo
import os, subprocess, sys

os.chdir('/kaggle/working/sr')

if not os.path.exists('pyproject.toml') and not os.path.exists('setup.py'):
    with open('setup.py', 'w') as f:
        f.write("from setuptools import setup, find_packages\nsetup(name='sessionrerank', version='0.1', packages=find_packages(where='src'), package_dir={'': 'src'})\n")
    print('[setup] wrote setup.py')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('[install pkg] done')

In [ ]:
# CELL 5: HF token from Kaggle secrets
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token
        print('[HF_TOKEN] set from secrets')
    else:
        print('[HF_TOKEN] empty in secrets')
except BaseException as e:
    print('[HF_TOKEN] not available:', e)

In [ ]:
# CELL A: Clean import state check
import sys
print('Python:', sys.version)
print('Executable:', sys.executable)

In [ ]:
# CELL B: Verify benchmark layout
from pathlib import Path

DATA_ROOT = Path('/kaggle/working/sr/data')
candidates = [
    DATA_ROOT / 'toolbench_hf' / 'benchmark',
    DATA_ROOT / 'benchmark',
    DATA_ROOT / 'toolbench' / 'benchmark',
]

BENCH = next((c for c in candidates if c.exists()), None)
print('Candidates:')
for c in candidates:
    print(c, '->', c.exists())

required = [
    'g1_instruction-00000-of-00001.parquet',
    'g1_category-00000-of-00001.parquet',
    'g1_tool-00000-of-00001.parquet',
]

if BENCH is not None:
    missing = [f for f in required if not (BENCH / f).exists()]
    print('Using:', BENCH)
    print('Missing:', missing)
else:
    print('No benchmark directory found — will download in CELL C')

In [ ]:
# CELL C: Download ToolBench (re-download if missing)
import os, subprocess, sys
from pathlib import Path

os.chdir('/kaggle/working/sr')

DATA_ROOT = Path('/kaggle/working/sr/data')
BENCH_CANDIDATES = [
    DATA_ROOT / 'toolbench_hf' / 'benchmark',
    DATA_ROOT / 'benchmark',
    DATA_ROOT / 'toolbench' / 'benchmark',
]

REQUIRED = [
    'g1_instruction-00000-of-00001.parquet',
    'g1_category-00000-of-00001.parquet',
    'g1_tool-00000-of-00001.parquet',
]

def bench_ok():
    for c in BENCH_CANDIDATES:
        if c.exists() and all((c / f).exists() for f in REQUIRED):
            return True
    return False

if bench_ok():
    print('[download] benchmark already present, skipping')
else:
    print('[download] fetching ToolBench from HuggingFace...')
    subprocess.run(
        [sys.executable, '-m', 'sessionrerank.data.download'],
        cwd='/kaggle/working/sr',
        check=True,
    )
    print('[download] complete')

# Re-verify
from pathlib import Path
bench = next((c for c in BENCH_CANDIDATES if c.exists()), None)
if bench:
    files = list(bench.glob('*.parquet'))
    print(f'[verify] benchmark at {bench}, {len(files)} parquet files')
else:
    raise FileNotFoundError('Benchmark not found after download — check HF_TOKEN and internet')

In [ ]:
# CELL 9: Run evaluation — T4-safe flags
import subprocess, sys, os, time

os.chdir('/kaggle/working/sr')

cmd = [
    sys.executable, '-m', 'sessionrerank.eval.run_all',
    '--device', 'cuda',
    '--encoder', 'sentence-transformers/all-MiniLM-L6-v2',
    '--seeds', '0,1,2',
    '--fit-sessions', '4000',
    '--cooc',
    '--cooc-sessions', '20000',
    '--cross-encoder',
    '--cross-encoder-model', 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    '--cross-encoder-topk', '100',
    '--gemma-rerank', 'google/gemma-2-2b-it',
    '--gemma-rerank-topk', '50',
    '--gemma-rerank-batch-size', '4',
    '--gemma-4bit',
    '--free-embed-after-encode',
    '--resume',
    '--out', 'results/metrics.json',
]

print('[run_all] starting...')
t0 = time.time()
r = subprocess.run(cmd, cwd='/kaggle/working/sr')
print('[run_all] exit', r.returncode, 'elapsed', round(time.time()-t0), 's')

In [ ]:
# CELL 10: Show metrics
import json, pathlib

result_path = pathlib.Path('/kaggle/working/sr/results/metrics.json')
if not result_path.exists():
    partial = result_path.with_suffix('.json.partial')
    if partial.exists():
        result_path = partial
        print('[results] showing partial checkpoint')

if result_path.exists():
    data = json.loads(result_path.read_text())
    splits = data.get('splits', {})
    if not splits and 'seeds' in data:
        splits = data['seeds'][0].get('splits', {}) if data['seeds'] else {}
    print(f"{'Split':<20} {'Regime':<10} {'Method':<52} {'NDCG@5':>8} {'Hit@5':>8}")
    print('-' * 104)
    for split, sblk in splits.items():
        for regime, rblk in sblk.get('regimes', {}).items():
            for method, metrics in rblk.items():
                if isinstance(metrics, dict) and 'NDCG@5' in metrics:
                    v = metrics['NDCG@5']
                    h = metrics['Hit@5']
                    nd = v['mean'] if isinstance(v, dict) else v
                    hi = h['mean'] if isinstance(h, dict) else h
                    print(f'{split:<20} {regime:<10} {method:<52} {nd:>8.4f} {hi:>8.4f}')
else:
    print('[results] no metrics.json found')